# XAI: VIBI End-to-End

## 1. Imports

In [ ]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.transforms import functional as TF

from sklearn.metrics import accuracy_score, f1_score




## 2. Shared Utilities (Unified Interface)

In [ ]:
# Fix random seeds for reproducible experiments.
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Automatically locate the project root (must contain `dataset/archive` and `outputs`).
def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / 'dataset' / 'archive').exists() and (p / 'outputs').exists():
            return p
    raise FileNotFoundError(' Graduation Thesis ')


class ResizeLongestSidePadSquare:
    def __init__(self, size: int, fill: int = 0):
        self.size = int(size)
        self.fill = int(fill)

    def __call__(self, img: Image.Image):
        w, h = img.size
        scale = self.size / max(w, h)
        nw = max(1, int(round(w * scale)))
        nh = max(1, int(round(h * scale)))
        img = TF.resize(img, [nh, nw], interpolation=transforms.InterpolationMode.BILINEAR)
        pw, ph = self.size - nw, self.size - nh
        left, top = pw // 2, ph // 2
        right, bottom = pw - left, ph - top
        return TF.pad(img, [left, top, right, bottom], fill=self.fill)


class ManifestDataset(Dataset):
    # Build samples from the split manifest and return `(image, label, rel_path)`.
    def __init__(self, dataset_root: Path, df: pd.DataFrame, class_to_idx: dict, transform=None):
        self.dataset_root = Path(dataset_root)
        self.transform = transform
        self.class_to_idx = dict(class_to_idx)
        self.samples = []
        for _, r in df.iterrows():
            rel = str(r['path'])
            cls = str(r['class'])
            if cls not in self.class_to_idx:
                continue
            p = self.dataset_root / rel
            if p.exists():
                self.samples.append((p, self.class_to_idx[cls], rel))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        p, y, rel = self.samples[i]
        x = Image.open(p).convert('RGB')
        if self.transform is not None:
            x = self.transform(x)
        return x, y, rel


def build_baseline(model_key: str, num_classes: int, use_pretrained: bool = False):
    key = model_key.lower()
    if key == 'resnet18':
        w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if use_pretrained else None
        m = torchvision.models.resnet18(weights=w)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if key == 'vgg16_bn':
        w = torchvision.models.VGG16_BN_Weights.IMAGENET1K_V1 if use_pretrained else None
        m = torchvision.models.vgg16_bn(weights=w)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m
    if key == 'vit_b_16':
        w = torchvision.models.ViT_B_16_Weights.IMAGENET1K_V1 if use_pretrained else None
        m = torchvision.models.vit_b_16(weights=w)
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
        return m
    if key == 'swin_t':
        w = torchvision.models.Swin_T_Weights.IMAGENET1K_V1 if use_pretrained else None
        m = torchvision.models.swin_t(weights=w)
        m.head = nn.Linear(m.head.in_features, num_classes)
        return m
    raise ValueError(f' baseline: {model_key}')


def normalize_map(m: np.ndarray) -> np.ndarray:
    m = m.astype(np.float32)
    m -= m.min()
    d = m.max() + 1e-8
    return m / d


def topk_mask(score_map: np.ndarray, ratio: float = 0.1) -> np.ndarray:
    h, w = score_map.shape
    flat = score_map.reshape(-1)
    k = max(1, int(round(len(flat) * ratio)))
    idx = np.argpartition(flat, -k)[-k:]
    mask = np.zeros_like(flat, dtype=np.float32)
    mask[idx] = 1.0
    return mask.reshape(h, w)


def iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a > 0, b > 0).sum()
    union = np.logical_or(a > 0, b > 0).sum()
    return float(inter / (union + 1e-8))


def deletion_aopc(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, steps=5) -> float:
    # Deletion-AOPC: progressively remove high-importance regions and measure the drop in target confidence.
    with torch.no_grad():
        p0 = torch.softmax(model(x), dim=1)[0, cls_idx].item()

    h, w = score_map.shape
    imp = score_map.reshape(-1)
    order = np.argsort(-imp)
    x_work = x.clone()
    drops = []
    total = h * w

    for s in range(1, steps + 1):
        cut = int(total * s / steps)
        idx = order[:cut]
        m = np.ones(total, dtype=np.float32)
        m[idx] = 0.0
        m = torch.from_numpy(m.reshape(1, 1, h, w)).to(x.device)
        if x_work.shape[-2:] != (h, w):
            m = torch.nn.functional.interpolate(m, size=x_work.shape[-2:], mode='nearest')
        x_mask = x * m
        with torch.no_grad():
            p = torch.softmax(model(x_mask), dim=1)[0, cls_idx].item()
        drops.append(p0 - p)

    return float(np.mean(drops))


def insertion_auc(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, steps=5) -> float:
    # Insertion-AUC: progressively insert high-importance regions and compute the area under the confidence curve.
    h, w = score_map.shape
    imp = score_map.reshape(-1)
    order = np.argsort(-imp)
    total = h * w
    xs = []
    ys = []

    for s in range(1, steps + 1):
        keep = int(total * s / steps)
        idx = order[:keep]
        m = np.zeros(total, dtype=np.float32)
        m[idx] = 1.0
        m = torch.from_numpy(m.reshape(1, 1, h, w)).to(x.device)
        if x.shape[-2:] != (h, w):
            m = torch.nn.functional.interpolate(m, size=x.shape[-2:], mode='nearest')
        x_ins = x * m
        with torch.no_grad():
            p = torch.softmax(model(x_ins), dim=1)[0, cls_idx].item()
        xs.append(s / steps)
        ys.append(p)

    return float(np.trapezoid(ys, xs))




def comprehensiveness(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, top_ratio=0.1) -> float:
    # Faithfulness: drop in confidence after removing important regions; higher is better.
    h, w = score_map.shape
    m = topk_mask(score_map, top_ratio)
    m = torch.from_numpy(m).float().view(1, 1, h, w).to(x.device)
    if x.shape[-2:] != (h, w):
        m = torch.nn.functional.interpolate(m, size=x.shape[-2:], mode='nearest')

    with torch.no_grad():
        p_full = torch.softmax(model(x), dim=1)[0, cls_idx].item()
        p_drop = torch.softmax(model(x * (1.0 - m)), dim=1)[0, cls_idx].item()
    return float(p_full - p_drop)


def sufficiency(model, x: torch.Tensor, cls_idx: int, score_map: np.ndarray, top_ratio=0.1) -> float:
    # Faithfulness: confidence loss when only important regions are kept; lower is better.
    h, w = score_map.shape
    m = topk_mask(score_map, top_ratio)
    m = torch.from_numpy(m).float().view(1, 1, h, w).to(x.device)
    if x.shape[-2:] != (h, w):
        m = torch.nn.functional.interpolate(m, size=x.shape[-2:], mode='nearest')

    with torch.no_grad():
        p_full = torch.softmax(model(x), dim=1)[0, cls_idx].item()
        p_keep = torch.softmax(model(x * m), dim=1)[0, cls_idx].item()
    return float(p_full - p_keep)


def selected_feature_ratio(score_map: np.ndarray, threshold: float = 0.5) -> float:
    # Complexity: selected-feature ratio; lower means sparser explanations.
    m = normalize_map(score_map)
    return float((m >= threshold).mean())


def sparsity_from_ratio(ratio: float) -> float:
    # Complexity: sparsity = 1 - selected-feature ratio; higher means sparser explanations.
    return float(1.0 - ratio)



def save_explanation_visual(x: torch.Tensor, score_map: np.ndarray, out_png: Path, mean, std):
    # Save a visualization that overlays the explanation heatmap on the input image.
    img = x.detach().cpu()[0].permute(1, 2, 0).numpy()
    mean = np.array(mean, dtype=np.float32).reshape(1, 1, 3)
    std = np.array(std, dtype=np.float32).reshape(1, 1, 3)
    img = np.clip(img * std + mean, 0.0, 1.0)
    h, w = img.shape[:2]

    hm = normalize_map(score_map)
    hm = np.array(Image.fromarray((hm * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR)).astype(np.float32) / 255.0

    heat = np.zeros((h, w, 3), dtype=np.float32)
    heat[..., 0] = hm
    heat[..., 2] = 1.0 - hm

    overlay = np.clip(0.65 * img + 0.35 * heat, 0.0, 1.0)

    out_png.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((overlay * 255).astype(np.uint8)).save(out_png)

def stability_iou(explain_fn, model, x: torch.Tensor, cls_idx: int, top_ratio=0.1, noise_std=0.03) -> float:
    base_map, _ = explain_fn(model, x, cls_idx)
    n = torch.clamp(x + torch.randn_like(x) * noise_std, -5, 5)
    new_map, _ = explain_fn(model, n, cls_idx)
    a = topk_mask(base_map, top_ratio)
    b = topk_mask(new_map, top_ratio)
    return iou(a, b)





## 3. Global Configuration

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_ROOT = PROJECT_ROOT / 'dataset'
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'data_prep'
SPLIT_ALL_PATH = SPLIT_DIR / 'split_all.csv'
PREPROCESS_YAML = SPLIT_DIR / 'preprocess_config.yaml'
PREPROCESS_JSON = SPLIT_DIR / 'preprocess_config.json'

# Default baseline (can be changed to `resnet18` / `vgg16_bn` / `vit_b_16` / `swin_t`).
BASELINE_KEY = 'swin_t'
MAX_SAMPLES = 0
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)

if PREPROCESS_YAML.exists():
    try:
        import yaml
        preprocess_cfg = yaml.safe_load(PREPROCESS_YAML.read_text(encoding='utf-8'))
    except Exception:
        preprocess_cfg = json.loads(PREPROCESS_JSON.read_text(encoding='utf-8')) if PREPROCESS_JSON.exists() else {}
elif PREPROCESS_JSON.exists():
    preprocess_cfg = json.loads(PREPROCESS_JSON.read_text(encoding='utf-8'))
else:
    preprocess_cfg = {}

IMG_SIZE = int(preprocess_cfg.get('img_size', 224))
PAD_FILL = int(preprocess_cfg.get('pad_fill', 0))
NORM_MEAN = tuple(preprocess_cfg.get('normalize', {}).get('mean', [0.485, 0.456, 0.406]))
NORM_STD = tuple(preprocess_cfg.get('normalize', {}).get('std', [0.229, 0.224, 0.225]))

split_df = pd.read_csv(SPLIT_ALL_PATH)
classes = sorted(split_df['class'].dropna().unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}

test_df = split_df[split_df['split'] == 'test'].copy().reset_index(drop=True)
if MAX_SAMPLES > 0:
    test_df = test_df.iloc[:MAX_SAMPLES].copy().reset_index(drop=True)

eval_tfms = transforms.Compose([
    ResizeLongestSidePadSquare(IMG_SIZE, fill=PAD_FILL),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

test_ds = ManifestDataset(DATASET_ROOT, test_df, class_to_idx, transform=eval_tfms)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

print('[INFO] baseline =', BASELINE_KEY)
print('[INFO] test samples =', len(test_ds))




## 4. Method Implementation

In [ ]:
# End-to-end VIBI with a Transformer token bottleneck.
METHOD_KEY = 'vibi_e2e'
VARIANT_KEY = 'end-to-end'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'xai' / 'end-to-end' / METHOD_KEY / BASELINE_KEY
TB_VIBI_CKPT = OUT_DIR / 'tb_vibi_best.pt'
FORCE_RETRAIN = False

if BASELINE_KEY != 'swin_t':
    raise ValueError("TB-VIBI is implemented as a Swin-T token-bottleneck variant in this notebook, so use BASELINE_KEY = 'swin_t' first.")

PATCH = 16
GRID = IMG_SIZE // PATCH
NUM_TOKENS = GRID * GRID
K = 8
TARGET_RATIO = K / float(NUM_TOKENS)

BASELINE_TRAIN_CFG = {
    'resnet18': {'batch_size': 128, 'epochs': 20, 'lr': 1e-3, 'weight_decay': 1e-4},
    'vgg16_bn': {'batch_size': 128, 'epochs': 20, 'lr': 1e-3, 'weight_decay': 1e-4},
    'vit_b_16': {'batch_size': 32, 'epochs': 20, 'lr': 1e-4, 'weight_decay': 0.05},
    'swin_t': {'batch_size': 32, 'epochs': 20, 'lr': 5e-5, 'weight_decay': 0.05},
}
BASE_CFG = BASELINE_TRAIN_CFG.get(BASELINE_KEY, BASELINE_TRAIN_CFG['swin_t'])

TOTAL_EPOCHS = int(BASE_CFG['epochs'])
WARMUP_EPOCHS = max(3, TOTAL_EPOCHS // 4)
JOINT_EPOCHS = max(1, TOTAL_EPOCHS - WARMUP_EPOCHS)
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = float(BASE_CFG['weight_decay'])

TAU_START = 1.0
TAU_MID = 0.5
TAU_END = 0.1

LAMBDA_KL = 0.005
LAMBDA_BUDGET = 0.10
LAMBDA_ENTROPY = 0.02
LAMBDA_SMOOTH = 0.02
LAMBDA_DROP = 4.0
LAMBDA_CONS = 0.05
CONS_NOISE_STD = 0.03

class TokenSelector(nn.Module):
    def __init__(self, feat_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        return self.net(tokens).squeeze(-1)


class TBVIBI(nn.Module):
    def __init__(self, num_classes: int, use_pretrained: bool = True):
        super().__init__()
        backbone = build_baseline('swin_t', num_classes, use_pretrained=use_pretrained)

        baseline_ckpt = PROJECT_ROOT / 'outputs' / 'baseline' / 'swin_t' / 'best.pt'
        if baseline_ckpt.exists():
            payload = torch.load(baseline_ckpt, map_location='cpu')
            backbone.load_state_dict(payload['model_state_dict'])
            print('[INFO] Initialized the TB-VIBI backbone from the baseline checkpoint ->', baseline_ckpt)
        else:
            print('[INFO] No baseline checkpoint found for the TB-VIBI backbone; using ImageNet pretrained weights instead.')

        self.backbone_features = backbone.features
        self.backbone_norm = backbone.norm
        self.feat_dim = int(backbone.head.in_features)
        self.selector = TokenSelector(self.feat_dim)
        self.classifier = nn.Linear(self.feat_dim, num_classes)
        self.classifier.load_state_dict(backbone.head.state_dict())

    def extract_tokens(self, x: torch.Tensor):
        z = self.backbone_features(x)
        z = self.backbone_norm(z)  # B, H, W, C
        b, h, w, c = z.shape
        tokens = z.view(b, h * w, c)
        return tokens, h, w

    def token_scores(self, x: torch.Tensor):
        tokens, h, w = self.extract_tokens(x)
        scores = self.selector(tokens)
        return tokens, scores, h, w

    def masked_pool(self, tokens: torch.Tensor, gates: torch.Tensor):
        denom = gates.sum(dim=1, keepdim=True).clamp_min(1e-6)
        pooled = (tokens * gates.unsqueeze(-1)).sum(dim=1) / denom
        return pooled

    def classify_with_gates(self, tokens: torch.Tensor, gates: torch.Tensor):
        pooled = self.masked_pool(tokens, gates)
        return self.classifier(pooled)

    def forward(self, x: torch.Tensor, tau: float = 0.1, hard: bool = True, return_extras: bool = False):
        tokens, scores, h, w = self.token_scores(x)
        probs = torch.sigmoid(scores / max(float(tau), 1e-4))
        hard_gates = hard_topk_tokens(scores, K)
        if hard:
            gates = hard_gates
        else:
            soft_sample = gumbel_sigmoid(scores, tau=tau, training=self.training)
            # Straight-through top-k: use a hard bottleneck in the forward pass and soft gates for gradient flow.
            gates = hard_gates + soft_sample - soft_sample.detach()
        logits = self.classify_with_gates(tokens, gates)
        if return_extras:
            return logits, {'scores': scores, 'gates': hard_gates, 'probs': probs, 'h': h, 'w': w}
        return logits


def gumbel_sigmoid(logits: torch.Tensor, tau: float, training: bool = True):
    if training:
        u = torch.rand_like(logits).clamp_(1e-6, 1 - 1e-6)
        noise = torch.log(u) - torch.log(1.0 - u)
        return torch.sigmoid((logits + noise) / max(float(tau), 1e-4))
    return torch.sigmoid(logits / max(float(tau), 1e-4))


def hard_topk_tokens(scores: torch.Tensor, k: int):
    b, n = scores.shape
    idx = torch.topk(scores, k=min(int(k), int(n)), dim=1).indices
    gates = torch.zeros_like(scores)
    gates.scatter_(1, idx, 1.0)
    return gates


def bernoulli_kl_to_prior(probs: torch.Tensor, rho: float):
    prior = torch.full_like(probs, float(rho))
    q = probs.clamp(1e-6, 1.0 - 1e-6)
    p = prior.clamp(1e-6, 1.0 - 1e-6)
    kl = q * torch.log(q / p) + (1.0 - q) * torch.log((1.0 - q) / (1.0 - p))
    return kl.mean()


def token_smoothness_loss(values: torch.Tensor, h: int, w: int):
    grid = values.view(values.size(0), h, w)
    dh = (grid[:, 1:, :] - grid[:, :-1, :]).abs().mean()
    dw = (grid[:, :, 1:] - grid[:, :, :-1]).abs().mean()
    return dh + dw


def anneal_tau(epoch_idx: int, total_epochs: int):
    if total_epochs <= 1:
        return TAU_END
    pos = epoch_idx / float(total_epochs - 1)
    if pos <= 0.5:
        alpha = pos / 0.5
        return TAU_START + alpha * (TAU_MID - TAU_START)
    alpha = (pos - 0.5) / 0.5
    return TAU_MID + alpha * (TAU_END - TAU_MID)


def set_backbone_phase(model: TBVIBI, warmup: bool):
    for p in model.backbone_features.parameters():
        p.requires_grad_(False)
    for p in model.backbone_norm.parameters():
        p.requires_grad_(False)

    if not warmup:
        modules = []
        if len(model.backbone_features) >= 2:
            modules.extend([model.backbone_features[-2], model.backbone_features[-1]])
        else:
            modules.append(model.backbone_features)
        modules.append(model.backbone_norm)
        for module in modules:
            for p in module.parameters():
                p.requires_grad_(True)


def build_optimizer(model: TBVIBI):
    head_params = list(model.selector.parameters()) + list(model.classifier.parameters())
    backbone_params = [p for p in list(model.backbone_features.parameters()) + list(model.backbone_norm.parameters()) if p.requires_grad]
    param_groups = [
        {'params': head_params, 'lr': HEAD_LR},
    ]
    if backbone_params:
        param_groups.append({'params': backbone_params, 'lr': BACKBONE_LR})
    return optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)


def token_map_to_image(mask_1d: torch.Tensor, h: int, w: int, mode: str = 'bilinear'):
    grid = mask_1d.view(mask_1d.size(0), 1, h, w)
    if mode == 'nearest':
        return torch.nn.functional.interpolate(grid, size=(IMG_SIZE, IMG_SIZE), mode='nearest')
    return torch.nn.functional.interpolate(grid, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)


train_df = split_df[split_df['split'] == 'train'].copy().reset_index(drop=True)
val_df = split_df[split_df['split'] == 'val'].copy().reset_index(drop=True)
train_ds = ManifestDataset(DATASET_ROOT, train_df, class_to_idx, transform=eval_tfms)
val_ds = ManifestDataset(DATASET_ROOT, val_df, class_to_idx, transform=eval_tfms)
train_loader = DataLoader(train_ds, batch_size=int(BASE_CFG['batch_size']), shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=int(BASE_CFG['batch_size']), shuffle=False, num_workers=0)

OUT_DIR.mkdir(parents=True, exist_ok=True)
tb_vibi = TBVIBI(len(classes), use_pretrained=True).to(DEVICE)

if TB_VIBI_CKPT.exists() and not FORCE_RETRAIN:
    payload = torch.load(TB_VIBI_CKPT, map_location=DEVICE)
    tb_vibi.load_state_dict(payload['model_state_dict'])
    print('[INFO] Loaded TB-VIBI checkpoint:', TB_VIBI_CKPT)
else:
    best_val_f1 = -1.0
    best_state = None
    total_epochs = WARMUP_EPOCHS + JOINT_EPOCHS
    optimizer = None

    for ep in range(total_epochs):
        warmup = ep < WARMUP_EPOCHS
        set_backbone_phase(tb_vibi, warmup=warmup)
        optimizer = build_optimizer(tb_vibi)
        tau = anneal_tau(ep, total_epochs)

        tb_vibi.train()
        losses = []
        cls_losses = []
        kl_losses = []
        budget_losses = []
        entropy_losses = []
        smooth_losses = []
        drop_losses = []
        cons_losses = []

        for x, y, _ in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            tokens, scores, h, w = tb_vibi.token_scores(x)
            probs = torch.sigmoid(scores / max(float(tau), 1e-4))
            soft_sample = gumbel_sigmoid(scores, tau=tau, training=True)
            hard_gates = hard_topk_tokens(scores, K)
            gates_st = hard_gates + soft_sample - soft_sample.detach()

            logits = tb_vibi.classify_with_gates(tokens, gates_st)
            logits_drop = tb_vibi.classify_with_gates(tokens, 1.0 - gates_st)

            l_cls = torch.nn.functional.cross_entropy(logits, y)
            l_kl = bernoulli_kl_to_prior(probs, TARGET_RATIO)
            l_budget = ((probs.mean(dim=1) - TARGET_RATIO) ** 2).mean()

            p = probs.clamp(1e-6, 1 - 1e-6)
            l_entropy = (-(p * torch.log(p) + (1.0 - p) * torch.log(1.0 - p))).mean()
            l_smooth = token_smoothness_loss(probs, h, w)
            l_drop = torch.softmax(logits_drop, dim=1).gather(1, y.view(-1, 1)).mean()

            with torch.no_grad():
                ref_probs = probs.detach()
            noisy_x = torch.clamp(x + torch.randn_like(x) * CONS_NOISE_STD, -5.0, 5.0)
            _, noisy_scores, _, _ = tb_vibi.token_scores(noisy_x)
            noisy_probs = torch.sigmoid(noisy_scores / max(float(tau), 1e-4))
            l_cons = torch.nn.functional.mse_loss(noisy_probs, ref_probs)

            loss = (
                l_cls +
                LAMBDA_KL * l_kl +
                LAMBDA_BUDGET * l_budget +
                LAMBDA_ENTROPY * l_entropy +
                LAMBDA_SMOOTH * l_smooth +
                LAMBDA_DROP * l_drop +
                LAMBDA_CONS * l_cons
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tb_vibi.parameters(), 1.0)
            optimizer.step()

            losses.append(float(loss.item()))
            cls_losses.append(float(l_cls.item()))
            kl_losses.append(float(l_kl.item()))
            budget_losses.append(float(l_budget.item()))
            entropy_losses.append(float(l_entropy.item()))
            smooth_losses.append(float(l_smooth.item()))
            drop_losses.append(float(l_drop.item()))
            cons_losses.append(float(l_cons.item()))

        tb_vibi.eval()
        ys, ps = [], []
        with torch.no_grad():
            for x, y, _ in val_loader:
                x = x.to(DEVICE)
                y = y.to(DEVICE)
                logits = tb_vibi(x, tau=tau, hard=True)
                pred = logits.argmax(1)
                ys.extend(y.cpu().numpy().tolist())
                ps.extend(pred.cpu().numpy().tolist())

        val_acc = accuracy_score(ys, ps) if ys else 0.0
        val_f1 = f1_score(ys, ps, average='macro') if ys else 0.0
        phase = 'warmup' if warmup else 'joint'
        print(
            f'[TB-VIBI][{ep+1:02d}/{total_epochs:02d}] phase={phase} tau={tau:.3f} '
            f'loss={np.mean(losses):.4f} cls={np.mean(cls_losses):.4f} '
            f'kl={np.mean(kl_losses):.4f} budget={np.mean(budget_losses):.4f} '
            f'ent={np.mean(entropy_losses):.4f} smooth={np.mean(smooth_losses):.4f} '
            f'drop={np.mean(drop_losses):.4f} cons={np.mean(cons_losses):.4f} '
            f'val_acc={val_acc:.4f} val_f1={val_f1:.4f}'
        )

        if val_f1 > best_val_f1:
            best_val_f1 = float(val_f1)
            best_state = {k: v.detach().cpu() for k, v in tb_vibi.state_dict().items()}

    if best_state is not None:
        tb_vibi.load_state_dict(best_state)

    torch.save(
        {
            'model_state_dict': tb_vibi.state_dict(),
            'best_val_f1': float(best_val_f1),
            'warmup_epochs': int(WARMUP_EPOCHS),
            'joint_epochs': int(JOINT_EPOCHS),
            'k': int(K),
            'target_ratio': float(TARGET_RATIO),
        },
        TB_VIBI_CKPT,
    )
    print('[OK] Saved TB-VIBI checkpoint:', TB_VIBI_CKPT)


tb_vibi.eval()


def explain_soft_fn(model_unused, x: torch.Tensor, cls_idx: int):
    # Return the soft token-importance map.
    t0 = time.time()
    with torch.no_grad():
        _, scores, h, w = tb_vibi.token_scores(x)
        probs = torch.sigmoid(scores / TAU_END)
        soft_map = token_map_to_image(probs, h, w, mode='bilinear')
    return soft_map[0, 0].detach().cpu().numpy().astype(np.float32), time.time() - t0


def explain_fn(model_unused, x: torch.Tensor, cls_idx: int):
    # Return the hard top-k token-selection map used as the discrete explanation.
    t0 = time.time()
    with torch.no_grad():
        _, scores, h, w = tb_vibi.token_scores(x)
        hard_tokens = hard_topk_tokens(scores, K)
        hard_map = token_map_to_image(hard_tokens, h, w, mode='nearest')
    return hard_map[0, 0].detach().cpu().numpy().astype(np.float32), time.time() - t0


def explain_full_pipeline(x: torch.Tensor):
    # Return the prediction together with both hard and soft explanation maps.
    t0 = time.time()
    with torch.no_grad():
        logits, extras = tb_vibi(x, tau=TAU_END, hard=True, return_extras=True)
        pred = int(logits.argmax(1).item())
        h = int(extras['h'])
        w = int(extras['w'])
        hard_map = token_map_to_image(extras['gates'], h, w, mode='nearest')
        soft_map = token_map_to_image(extras['probs'], h, w, mode='bilinear')
    return pred, hard_map[0, 0].detach().cpu().numpy().astype(np.float32), soft_map[0, 0].detach().cpu().numpy().astype(np.float32), time.time() - t0


baseline_model = tb_vibi



## 5. Unified Quantitative Evaluation and Outputs

In [ ]:
VIS_SAVE_N = 20
VIS_DIR = OUT_DIR / 'visuals'
VIS_DIR.mkdir(parents=True, exist_ok=True)

records = []
for i, (x, y, rel) in enumerate(test_loader):
    sample_t0 = time.time()
    x = x.to(DEVICE)
    y = y.to(DEVICE)

    pred, score_map, soft_map, t_explain = explain_full_pipeline(x)
    if i < VIS_SAVE_N:
        stem = Path(rel[0]).stem
        hard_vis_path = VIS_DIR / f"{i:04d}_{stem}_hard.png"
        soft_vis_path = VIS_DIR / f"{i:04d}_{stem}_soft.png"
        save_explanation_visual(x, score_map, hard_vis_path, NORM_MEAN, NORM_STD)
        save_explanation_visual(x, soft_map, soft_vis_path, NORM_MEAN, NORM_STD)

    aopc = deletion_aopc(baseline_model, x, pred, score_map)
    auci = insertion_auc(baseline_model, x, pred, score_map)
    siou = stability_iou(explain_fn, baseline_model, x, pred)
    comp = comprehensiveness(baseline_model, x, pred, score_map)
    suff = sufficiency(baseline_model, x, pred, score_map)
    sfr = selected_feature_ratio(score_map, threshold=0.5)
    spr = sparsity_from_ratio(sfr)
    sample_elapsed = time.time() - sample_t0
    print(f"[{i+1:03d}/{len(test_loader):03d}] explain done | explain={t_explain:.3f}s | total={sample_elapsed:.3f}s | aopc={aopc:.4f} | auc={auci:.4f}")

    records.append({
        'idx': int(i),
        'path': rel[0],
        'y_true': int(y.item()),
        'y_pred': int(pred),
        'explain_time_sec': float(t_explain),
        'aopc_deletion': float(aopc),
        'auc_insertion': float(auci),
        'iou_top10': float(siou),
        'comprehensiveness': float(comp),
        'sufficiency': float(suff),
        'selected_feature_ratio': float(sfr),
        'sparsity': float(spr),
    })

res_df = pd.DataFrame(records)

summary = {
    'method': METHOD_KEY,
    'variant': VARIANT_KEY,
    'baseline_key': BASELINE_KEY,
    'num_samples': int(len(res_df)),
    'metrics': {
        'mean_explain_time_sec': float(res_df['explain_time_sec'].mean()) if len(res_df) else None,
        'median_explain_time_sec': float(res_df['explain_time_sec'].median()) if len(res_df) else None,
        'mean_aopc_deletion': float(res_df['aopc_deletion'].mean()) if len(res_df) else None,
        'mean_auc_insertion': float(res_df['auc_insertion'].mean()) if len(res_df) else None,
        'mean_iou_top10': float(res_df['iou_top10'].mean()) if len(res_df) else None,
        'mean_comprehensiveness': float(res_df['comprehensiveness'].mean()) if len(res_df) else None,
        'mean_sufficiency': float(res_df['sufficiency'].mean()) if len(res_df) else None,
        'mean_selected_feature_ratio': float(res_df['selected_feature_ratio'].mean()) if len(res_df) else None,
        'mean_sparsity': float(res_df['sparsity'].mean()) if len(res_df) else None,
    }
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
(res_df).to_csv(OUT_DIR / 'per_sample_metrics.csv', index=False, encoding='utf-8')
(OUT_DIR / 'quant_metrics.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('[OK] saved:', OUT_DIR)
print(json.dumps(summary, indent=2, ensure_ascii=False))

